# 16. Trajectory-关联差异基因分析

沿 pseudotime 轨迹检测表达动态变化的基因——不依赖离散的 cluster 标签，直接回答
"沿 Chief→SPEM→IM 转化轴，哪些基因在哪个时间点启动/关闭"。

**方法**: 对每个基因拟合 `expression ~ smooth(pseudotime)` 的 GAM（广义加性模型），
用 `scipy.interpolate.UnivariateSpline` 做非参数平滑拟合，以 R² 量化拟合优度，
避免 statsmodels GAM 的安装依赖问题。

**输入**: 含 pseudotime 列的 h5ad（通常来自 `10_pseudotime` 或 06c subset 后的 pseudotime）

**输出**: 按 R² 排序的基因表 + 关键基因趋势图 + pathway 富集（可选）

## 生物学背景

**为什么做 Trajectory-关联 DEG？** 传统的 cluster-关联 DEG（`07_deg.ipynb`）
将细胞归入离散的 cluster，比较 cluster A vs B。但分化过程是连续的——
SPEM 细胞和早期肠化细胞共享部分转录程序，硬性分箱会丢失中间态信号。

Trajectory-关联 DEG 直接建模 expression = f(pseudotime)，不依赖离散标签，
能够：
1. 识别在分化过程中逐步激活/抑制的基因（而非仅在 cluster 边界跳跃）
2. 发现"拐点基因"——在特定 pseudotime 位置表达剧烈变化的基因
3. 分离早/中/晚期分化程序，为干预靶点筛选提供时间窗口信息

**方法选择**：GAM（广义加性模型）是非线性回归，用平滑样条拟合
expression~pseudotime 关系，检验平滑项是否显著。与简单的 Pearson/Spearman
相关相比，GAM 能捕获非单调变化（如先升后降的瞬时表达基因），更贴合
分化过程中基因表达的动力学特征。

**计算策略**：全基因组建模在实际数据集中计算量极大（数万基因 × 数万细胞）。
本 notebook 默认仅测试前 `N_TOP_GENES` 个高变基因（HVG），速度合理。
设 `N_TOP_GENES=None` 可测试全部基因（仅建议在 <5000 基因的小数据集中使用）。

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH      -- 含 pseudotime 列的 h5ad（来自 10_pseudotime 或 06c subset 后）
# OUTPUT_DIR          -- 产出目录
# PSEUDOTIME_COL      -- pseudotime 列名；None=自动检测（Monocle3 > CellRank2 > DPT > 通用 pseudotime）
# N_TOP_GENES         -- 测试前 N 个高变基因（加速；设 None 测试全部 HVG）
# FDR_THRESHOLD       -- 显著性阈值（R² threshold，非 FDR 因为本实现用 R² 代替 p-value）
# N_SPLINES           -- UnivariateSpline 样条基函数数（k=3 为三次样条；越大越灵活但易过拟合）
# HIGHLIGHT_GENES     -- 关键基因可视化列表（叠加在全基因组结果上）

UPSTREAM_PATH = "results/10_pseudotime_v1.h5ad"  # 或 06c subset 后的 pseudotime 产物
OUTPUT_DIR = "results/figures/16_trajectory_de"

PSEUDOTIME_COL = None  # None = 自动检测（Monocle3 > CellRank2 > DPT > 通用 pseudotime）
N_TOP_GENES = 500      # 测试前 N 个高变基因（加速；设 None 测试全部 HVG）
R2_THRESHOLD = 0.05    # R² 阈值（替代 FDR；R² < 此值视为不显著）
N_SPLINES = 3          # UnivariateSpline 次数（1=线性, 2=二次, 3=三次；越大越灵活）

# 关键基因可视化列表（叠加在全基因组结果上）
HIGHLIGHT_GENES = ["PGA3", "GIF", "LIPF", "TFF2", "WFDC2", "CDX2", "MUC2", "TFF3", "MKI67", "LGR5"]

OUTPUT_VERSION = 1
RANDOM_SEED = 42

In [ ]:
# === Setup：sys.path + 导入 + 加载上游 ===
# 多级回退策略：nbconvert/conda run 的 CWD 不稳定，
# 先试 CWD，再试从 notebooks/07_downstream/ 回退两级，最后用 notebook 自身路径推算。
import os, sys, gc
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline

_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")
print(f"src 存在: {os.path.isdir(os.path.join(_root, 'src', 'scrna_integration'))}")

import scanpy as sc
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
np.random.seed(RANDOM_SEED)

print(f"scanpy {sc.__version__}  |  numpy {np.__version__}")

# 加载上游数据
# 契约：需包含 pseudotime 列（来自 10_pseudotime、10b Monocle3、10c CellRank2 或 06c 的 DPT）
print(f"加载上游: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")

# 自动检测 pseudotime 列（按优先级：Monocle3 > CellRank2 > DPT > 通用 pseudotime）
# 为什么这个优先级？Monocle3 自动学习轨迹拓扑，生物学解释力最强；
# CellRank2 基于 fate mapping 推断终末状态间 pseudotime，估计稳健；
# DPT 是 scanpy 内置，无外部依赖；"pseudotime" 是通用 fallback。
if PSEUDOTIME_COL is None:
    for _cand in ["pseudotime_monocle3_v1", "cellrank2_pseudotime",
                   "ct_pseudotime", "dpt_pseudotime", "pseudotime"]:
        if _cand in adata.obs.columns and adata.obs[_cand].notna().sum() > 50:
            PSEUDOTIME_COL = _cand
            break
if PSEUDOTIME_COL is None:
    raise ValueError(
        f"未找到可用 pseudotime 列。可用 obs 列: {list(adata.obs.columns)}。"
        "请先运行 10_pseudotime 或手动指定 PSEUDOTIME_COL"
    )

print(f"Pseudotime 列: {PSEUDOTIME_COL} ({adata.obs[PSEUDOTIME_COL].notna().sum():,} 有效值)")

# 过滤无 pseudotime 值的细胞
_valid = adata.obs[PSEUDOTIME_COL].notna()
if _valid.sum() < adata.n_obs:
    print(f"  过滤 {(~_valid).sum()} 个无 pseudotime 值的细胞")
    adata = adata[_valid].copy()

### 方法: 非参数平滑拟合 + R² 评估

对每个基因 g，用 `UnivariateSpline` 拟合平滑曲线：
`expression_g ~ spline(pseudotime, k=N_SPLINES)`

其中 spline() 是三次平滑样条函数。用 R² 量化拟合优度——
R² 越高，基因表达与 pseudotime 的关系越强（越可能是 trajectory-关联基因）。

**为什么用 R² 而非 p-value？** `UnivariateSpline` 不直接提供显著性检验，
而 statsmodels GAM 安装依赖重（patsy + scipy 版本冲突常见）。
R² 作为效应量指标（而非显著性检验指标）在轨迹分析中同样有效：
R² 高的基因意味着其表达变化中有较大比例可被 pseudotime 解释，
这些基因优先于低 R² 基因进行下游分析。

**局限性**：R² 不区分线性/非线性关系——单调上升的管家基因也可能有高 R²，
但并非真正的"轨迹驱动"基因。下游解读需结合生物学知识过滤。
可选的改进方向：用 permutation test 打乱 pseudotime 多次拟合获得经验 null 分布，
计算 empirical p-value。此功能留作后续迭代。

In [ ]:
# === Trajectory-关联 DEG：UnivariateSpline 拟合 + R² 排序 ===
# 对每个基因沿 pseudotime 排序后的表达值拟合样条曲线，
# 计算 R² = 1 - SS_res / SS_tot，按 R² 降序输出 trajectory-关联基因。
# 复杂度 O(N_genes × N_cells)，使用预排序索引避免重复排序。

_pt = adata.obs[PSEUDOTIME_COL].values
_sort_idx = np.argsort(_pt)
_pt_sorted = _pt[_sort_idx]

# 选择测试基因：优先用 HVG（高变基因），减少计算量
# 为什么用 HVG？低表达/低变异基因的 R² 噪音大且生物学意义弱，
# 在轨迹分析中优先高变异基因是标准做法。
if N_TOP_GENES and "highly_variable" in adata.var.columns:
    _test_genes = adata.var_names[adata.var["highly_variable"]][:N_TOP_GENES].tolist()
    print(f"从 HVG 中取前 {N_TOP_GENES} 个基因")
else:
    _test_genes = adata.var_names.tolist()
    print(f"测试全部 {len(_test_genes)} 个基因（N_TOP_GENES=None 或 HVG 不存在）")

# 确保 HIGHLIGHT_GENES 在测试列表中
_n_added = 0
for g in HIGHLIGHT_GENES:
    if g in adata.var_names and g not in _test_genes:
        _test_genes.append(g)
        _n_added += 1
if _n_added:
    print(f"  追加 {_n_added} 个 HIGHLIGHT_GENES 至测试列表")

print(f"测试 {len(_test_genes)} 个基因...")

np.random.seed(RANDOM_SEED)
_results = []

for i, gene in enumerate(_test_genes):
    if i % 100 == 0:
        print(f"  进度: {i}/{len(_test_genes)}")

    # 提取该基因的表达向量（按 pseudotime 排序）
    _expr = adata[:, gene].X
    if sp.issparse(_expr):
        _expr = _expr.toarray().flatten()
    else:
        _expr = _expr.flatten()
    _expr_sorted = _expr[_sort_idx]

    # 拟合平滑样条曲线
    # k=3 为三次样条（默认），对大多数基因的表达轨迹足够灵活；
    # s=len(_pt_sorted) 控制平滑度——s 越大曲线越平滑（接近线性），
    # 设 s=n 让样条在保持足够拟合度前提下避免过拟合单细胞噪音。
    try:
        _spline = UnivariateSpline(_pt_sorted, _expr_sorted, k=min(N_SPLINES, 3), s=len(_pt_sorted))
        _fitted = _spline(_pt_sorted)

        # 计算 R²（拟合 vs 均值）：衡量 pseudotime 能解释多少表达变异
        _ss_res = np.sum((_expr_sorted - _fitted) ** 2)
        _ss_tot = np.sum((_expr_sorted - _expr_sorted.mean()) ** 2)
        _r2 = 1 - _ss_res / max(_ss_tot, 1e-10)

        # 计算拟合曲线的变化范围（fitted max - min）
        _range = float(_fitted.max() - _fitted.min())

        # 趋势方向（正 = 沿 pseudotime 表达上升，负 = 下降）
        # 比较拟合曲线前 100 个点的均值 vs 后 100 个点的均值，
        # 判定基因是"激活"（up）还是"抑制"（down）趋势。
        _n_tail = min(100, max(10, len(_fitted) // 50))
        _direction = "up" if _fitted[-_n_tail:].mean() > _fitted[:_n_tail].mean() else "down"

        _results.append({
            "gene": gene,
            "r_squared": float(_r2),
            "expression_range": _range,
            "direction": _direction,
            "mean_expression": float(_expr.mean()),
        })
    except Exception:
        # 跳过拟合失败的基因（如全零表达、常数表达等退化情况）
        continue

_df_results = pd.DataFrame(_results)
_df_results = _df_results.sort_values("r_squared", ascending=False).reset_index(drop=True)

print(f"\n拟合完成: {len(_df_results)} 基因")
print(f"R² >= {R2_THRESHOLD}: {(_df_results['r_squared'] >= R2_THRESHOLD).sum()} 基因")
print(f"\nTop 20 trajectory-关联基因（R² 排序）:")
print(_df_results.head(20).to_string(index=False))

In [ ]:
# === 保存完整结果表 + Top 基因趋势图 ===
# 保存 CSV 供下游分析和手工检查；绘制 Top + highlight 基因的趋势子图。

# 保存完整结果表
_csv_path = os.path.join(OUTPUT_DIR, "trajectory_de_results.csv")
_df_results.to_csv(_csv_path, index=False)
print(f"保存: {_csv_path} ({len(_df_results)} 基因)")

# 选择绘图基因：Top 12（R² 最高）+ HIGHLIGHT_GENES（即使 R² 低也值得看）
_plot_genes = list(_df_results.head(12)["gene"])
for g in HIGHLIGHT_GENES:
    if g in adata.var_names and g not in _plot_genes:
        _plot_genes.append(g)
_plot_genes = _plot_genes[:16]  # 最多 16 个子图（4×4 布局）

# 绘制子图网格
_ncols = 4
_nrows = (len(_plot_genes) + _ncols - 1) // _ncols
fig, axes = plt.subplots(_nrows, _ncols, figsize=(16, 3 * _nrows), squeeze=False)

for i, gene in enumerate(_plot_genes):
    ax = axes[i // _ncols, i % _ncols]

    # 提取该基因的表达并按 pseudotime 排序
    _expr = adata[:, gene].X
    if sp.issparse(_expr):
        _expr = _expr.toarray().flatten()
    else:
        _expr = _expr.flatten()
    _expr_sorted = _expr[_sort_idx]

    # 滑动窗口平滑（5% 窗口，至少 20 个细胞）
    _window = max(20, int(len(_expr_sorted) * 0.05))
    _smoothed = pd.Series(_expr_sorted).rolling(
        window=_window, center=True, min_periods=1
    ).mean()

    # 灰色散点为单细胞，红色曲线为滑动平均
    ax.scatter(_pt_sorted, _expr_sorted, s=0.3, alpha=0.05, color="gray")
    ax.plot(_pt_sorted, _smoothed.values, color="red", linewidth=2)

    # R² 标注（如果该基因在结果表中）
    _gene_r2 = _df_results.loc[_df_results["gene"] == gene, "r_squared"]
    _r2_text = f"R\u00b2={_gene_r2.values[0]:.3f}" if len(_gene_r2) > 0 else ""
    ax.set_title(f"{gene}\n{_r2_text}", fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("")

# 隐藏多余子图
for i in range(len(_plot_genes), _nrows * _ncols):
    axes[i // _ncols, i % _ncols].set_visible(False)

plt.suptitle(
    f"Trajectory-关联基因表达趋势（{PSEUDOTIME_COL}）\n"
    f"灰点 = 单细胞  |  红线 = 滑动平均  |  R\u00b2 = 拟合优度",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
_fig_path = os.path.join(OUTPUT_DIR, "trajectory_de_top_genes.png")
plt.savefig(_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"保存: {_fig_path}")

In [ ]:
# === Pathway 富集（可选，需 gseapy）===
# 将高 R² 基因分上升/下降两组，分别做 GO Biological Process 富集分析。
# 为什么分上升/下降？沿 pseudotime 表达上升的基因可能参与分化终末程序
# （如肠化相关通路），下降的基因可能代表祖细胞功能丢失。
# 分别富集有助于区分正向和负向驱动分化进程的功能模块。

_up_genes = _df_results[
    (_df_results["r_squared"] >= R2_THRESHOLD) & (_df_results["direction"] == "up")
]["gene"].tolist()
_down_genes = _df_results[
    (_df_results["r_squared"] >= R2_THRESHOLD) & (_df_results["direction"] == "down")
]["gene"].tolist()

print(f"R\u00b2 >= {R2_THRESHOLD} 上升基因: {len(_up_genes)}")
print(f"R\u00b2 >= {R2_THRESHOLD} 下降基因: {len(_down_genes)}")

try:
    import gseapy
    
    if _up_genes:
        _enr_up = gseapy.enrichr(
            gene_list=_up_genes[:200], gene_sets="GO_Biological_Process_2021",
            organism="human", outdir=None, no_plot=True,
        )
        if hasattr(_enr_up, "results") and len(_enr_up.results) > 0:
            print(f"\n上升基因 GO 富集 Top 10:")
            _cols = ["Term", "Adjusted P-value", "Overlap", "Genes"]
            _show_cols = [c for c in _cols if c in _enr_up.results.columns]
            print(_enr_up.results.head(10)[_show_cols].to_string(index=False))
    
    if _down_genes:
        _enr_down = gseapy.enrichr(
            gene_list=_down_genes[:200], gene_sets="GO_Biological_Process_2021",
            organism="human", outdir=None, no_plot=True,
        )
        if hasattr(_enr_down, "results") and len(_enr_down.results) > 0:
            print(f"\n下降基因 GO 富集 Top 10:")
            _cols = ["Term", "Adjusted P-value", "Overlap", "Genes"]
            _show_cols = [c for c in _cols if c in _enr_down.results.columns]
            print(_enr_down.results.head(10)[_show_cols].to_string(index=False))
except ImportError:
    print("gseapy 未安装，跳过 pathway 富集。安装: pip install gseapy")
except Exception as e:
    print(f"Pathway 富集异常: {e}")

In [ ]:
# === 释放内存 ===
del adata
gc.collect()
print("内存已释放。")